<a href="https://colab.research.google.com/github/ahmed8809/FlyRank_intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import duckdb
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



**Unit of analysis**

One row represents the daily performance of one content item (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).

**Tables**

This notebook uses the `fact_content_daily_performance` table for daily search and analytics metrics. The lane may later join `dim_content` to add static content metadata for clustering.

**Time window**

This notebook analyzes the partition `month='2026-03'`, a mid-panel month recommended for feature engineering.

**Learning objective**

This lane performs **unsupervised clustering**, so there is **no prediction label**. The objective is to discover recurring performance archetypes across content.

**Excluded fields**

`client_hash_id` and `content_hash_id` are excluded from clustering because they are identifiers rather than behavioral features.

In [5]:
query = f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
);
"""

con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

### Features

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

These are historical performance metrics available before clustering.

### Label

None.

This lane is unsupervised clustering, therefore no prediction label is used.

### Context

- report_date
- client_hash_id
- content_hash_id

Used for grouping and identification only.

### Excluded

- client_hash_id
- content_hash_id

Excluded because identifiers should not influence clustering.

In [6]:
feature_query = f"""
SELECT

    report_date,
    client_hash_id,
    content_hash_id,

    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_engaged_sessions

FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
    gsc_data_available IS TRUE
    AND ga4_data_available IS TRUE

LIMIT 20;
"""

features = con.sql(feature_query).df()

features

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,1,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,2,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,2,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,1,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,1,0
5,2026-03-01,client_65de48885f4ef01b,content_872342e050545a12,39,0,6.538462,1,0
6,2026-03-01,client_65de48885f4ef01b,content_3c286ded8bd68120,88,1,8.431818,1,0
7,2026-03-01,client_65de48885f4ef01b,content_b2108e8fe3360fa6,40,1,5.300000,1,0
8,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,0,30.304348,2,0
9,2026-03-01,client_65de48885f4ef01b,content_bd07be40ea0d5f54,23,0,5.478261,1,0


## 3. Verify it with queries (grain, counts, missing values, windows)


This section verifies three assumptions about the dataset.

1. The grain is one row per report date, client, and content item.
2. The selected partition contains the expected date range and row count.
3. Only rows with available GSC and GA4 data are retained by filtering with `IS TRUE`.

In [8]:
# Query 1 : Grain

grain = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS cnt

FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)

GROUP BY
    report_date,
    client_hash_id,
    content_hash_id

HAVING COUNT(*) > 1

LIMIT 10;
""").df()

print("Duplicate grain rows:")
display(grain)






FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows:


,report_date,client_hash_id,content_hash_id,cnt


In [12]:
count_span = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
);
""").df()

print("Row count and date span")
display(count_span)

Row count and date span


,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [10]:
availability = con.sql(f"""
SELECT
COUNT(*) available_rows

FROM read_parquet(
'{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE
gsc_data_available IS TRUE
AND ga4_data_available IS TRUE;
""").df()

print("Available rows")
display(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Available rows


,available_rows
0,364347


## 4. Data limits

### Feature availability

Only rows where both GSC and GA4 data are available are used.

### Limited history

Clients have different history lengths, so comparisons across clients should be interpreted carefully.

### No article text

The warehouse does not include article text, therefore this lane performs **structured metric clustering**, not semantic clustering.

### Limitation

Clusters describe recurring patterns in the available metrics. They should be interpreted as useful behavioral archetypes rather than true categories.

In [11]:
import pandas as pd

feature_description = pd.DataFrame({

    "Feature":[
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "ga4_engaged_sessions"
    ],

    "Available when?":[
        "Historical search impressions observed before clustering.",
        "Historical search clicks observed before clustering.",
        "Historical average search ranking before clustering.",
        "Historical pageviews already recorded.",
        "Historical engagement collected before clustering."
    ]
})

feature_description

,Feature,Available when?
0,gsc_impressions,Historical search impressions observed before ...
1,gsc_clicks,Historical search clicks observed before clust...
2,gsc_avg_position,Historical average search ranking before clust...
3,ga4_pageviews,Historical pageviews already recorded.
4,ga4_engaged_sessions,Historical engagement collected before cluster...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.